In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
sys.path.append("..")
from data.fetch_data import fetch_data

df = fetch_data()
df.head()

[INFO] Kaggle'dan veri indiriliyor...
Dataset URL: https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset
License(s): copyright-authors


  0%|          | 0.00/67.4k [00:00<?, ?B/s]


[INFO] Veri yüklendi: 5110 satır, 12 sütun


100%|██████████| 67.4k/67.4k [00:00<00:00, 329kB/s]


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [ ]:
fig = px.pie(
    df,
    names='stroke',
    title='Target Variable Distribution (Stroke)',
    color_discrete_sequence=['#4A90D9', '#E8E8E8']
)
fig.show()

In [12]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percentage': missing_pct
}).query('`Missing Count` > 0')

print(missing_df)

     Missing Count  Missing Percentage
bmi            201            3.933464


In [31]:
df['bmi_missing'] = df['bmi'].isnull().astype(int)

print(df.groupby('bmi_missing')['stroke'].mean())
print()
print(df.groupby('bmi_missing')['age'].mean())

bmi_missing
0    0.042575
1    0.199005
Name: stroke, dtype: float64

bmi_missing
0    42.865374
1    52.049154
Name: age, dtype: float64


In [17]:
print("Duplicate rows:", df.duplicated().sum())
print()
print("Unique values per column:")
print(df.nunique())

Duplicate rows: 0

Unique values per column:
id                   5110
gender                  3
age                   104
hypertension            2
heart_disease           2
ever_married            2
work_type               5
Residence_type          2
avg_glucose_level    3979
bmi                   418
smoking_status          4
stroke                  2
bmi_missing             2
dtype: int64


In [18]:
print(df['gender'].value_counts())

gender
Female    2994
Male      2115
Other        1
Name: count, dtype: int64


In [20]:
print(df['smoking_status'].value_counts())

smoking_status
never smoked       1892
Unknown            1544
formerly smoked     885
smokes              789
Name: count, dtype: int64


In [21]:
fig = px.box(
    df,
    y=['age', 'avg_glucose_level', 'bmi'],
    title='Numerical Variables Distribution',
)
fig.show()

In [22]:
print(df['bmi'].describe())
print()
print("BMI > 60 olan kişi sayısı:", (df['bmi'] > 60).sum())

count    4909.000000
mean       28.893237
std         7.854067
min        10.300000
25%        23.500000
50%        28.100000
75%        33.100000
max        97.600000
Name: bmi, dtype: float64

BMI > 60 olan kişi sayısı: 13


In [23]:
print(df['avg_glucose_level'].describe())
print()
print("Glucose > 200 olan kişi sayısı:", (df['avg_glucose_level'] > 200).sum())

count    5110.000000
mean      106.147677
std        45.283560
min        55.120000
25%        77.245000
50%        91.885000
75%       114.090000
max       271.740000
Name: avg_glucose_level, dtype: float64

Glucose > 200 olan kişi sayısı: 434


In [24]:
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

for col in categorical_cols:
    stroke_rate = df.groupby(col)['stroke'].mean().reset_index()
    stroke_rate.columns = [col, 'stroke_rate']
    
    fig = px.bar(
        stroke_rate,
        x=col,
        y='stroke_rate',
        title=f'Stroke Rate by {col}',
        labels={col: col, 'stroke_rate': 'Stroke Rate'}
    )
    fig.show()

In [25]:
fig = px.box(
    df,
    x='stroke',
    y='age',
    title='Age vs Stroke',
    labels={'stroke': 'Stroke', 'age': 'Age'}
)
fig.show()

fig = px.box(
    df,
    x='stroke',
    y='avg_glucose_level',
    title='Glucose Level vs Stroke',
    labels={'stroke': 'Stroke', 'avg_glucose_level': 'Avg Glucose Level'}
)
fig.show()

fig = px.box(
    df,
    x='stroke',
    y='bmi',
    title='BMI vs Stroke',
    labels={'stroke': 'Stroke', 'bmi': 'BMI'}
)
fig.show()

In [26]:
corr_cols = ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease', 'stroke']

corr_matrix = df[corr_cols].corr()

fig = px.imshow(
    corr_matrix,
    title='Correlation Matrix',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    text_auto='.2f'
)
fig.show()

In [29]:
for col in ['hypertension', 'heart_disease']:
    stroke_rate = df.groupby(col)['stroke'].mean().reset_index()
    stroke_rate.columns = [col, 'stroke_rate']
    stroke_rate[col] = stroke_rate[col].astype(str)
    
    fig = px.bar(
        stroke_rate,
        x=col,
        y='stroke_rate',
        title=f'Stroke Rate by {col}',
        labels={col: col, 'stroke_rate': 'Stroke Rate'}
    )
    fig.show()

In [30]:
df['bmi_missing'] = df['bmi'].isnull().astype(int)

print(df.groupby('bmi_missing')['stroke'].mean())
print()
print(df.groupby('bmi_missing')['age'].mean())

bmi_missing
0    0.042575
1    0.199005
Name: stroke, dtype: float64

bmi_missing
0    42.865374
1    52.049154
Name: age, dtype: float64
